# 第7回　データの整理と比較：クロス集計
## ―― 全体で見ると差があるのに、グループごとに見ると差が消えるのはなぜか

情報活用Ⅰ　／　北星学園大学　2026年度後期

今日つくる表は **クロス集計表（分割表）**。2つの項目を同時に数えた表である。
これはこの授業の中核であり、あなたの報告書の中心になる表でもある。

In [ ]:
# 準備：ライブラリと、練習用データ（北辰大学の学生400人）を読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語が豆腐（□）にならないようにする

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    # ネットから取れないときは、同じデータをその場で作る（中身は気にしなくてよい）
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan

print("読み込めた行数:", len(df))
df.head()

---
## 0. なぜピボットテーブルを使わないのか

Excelには「ピボットテーブル」という、同じ表を作る便利な機能がある。**それは使わない。**
作る表は同じである。使わないのは機能のほうだ。理由は1つ。

> **何をどう数えたかが記録されないため、あとから再現できず、他人も検証できない。**

ピボットテーブルは、マウスで列をドラッグして作る。半年後に「この表、どうやって作ったんですか」と聞かれても、**操作の履歴はどこにも残っていない**。
コードなら、どの列を、どの条件で、どう数えたかが全部残る。もう一度実行すれば同じ表が出る。

**結果だけが出てきて過程が残らない道具は、検証に使えない。**
これは、AIの出力に対してこの授業がとる態度と、まったく同じ構造である。

---
## 1. 1973年、カリフォルニア大学バークレー校

大学院の合格率に男女差がある、と大問題になった事件がある。まず全体の数字を見る。

In [ ]:
# 1973年 バークレー校 大学院 上位6学部の出願・合格データ
# 出典: Bickel, Hammel & O'Connell (1975) Science, 187(4175), 398-404.
berkeley = pd.DataFrame({
    "学部": ["A","A","B","B","C","C","D","D","E","E","F","F"],
    "性別": ["男","女"]*6,
    "出願": [825,108, 560,25, 325,593, 417,375, 191,393, 373,341],
    "合格": [512, 89, 353,17, 120,202, 138,131,  53, 94,  22, 24],
})
berkeley["不合格"] = berkeley["出願"] - berkeley["合格"]

# 全体の合格率
total = berkeley.groupby("性別")[["出願","合格"]].sum()
total["合格率"] = (total["合格"] / total["出願"] * 100).round(1)
total

**男性 44.5%、女性 30.4%。** 14ポイントもの差がある。

（よく引用される「男44%・女35%」は大学全体の数字。ここで見ているのは出願者の多い上位6学部で、差はさらに大きい。）

これだけ見れば「女性が不利に扱われている」と読める。
ところが、**学部ごとに分けて**数え直すと、話が変わる。

In [ ]:
# 学部ごとの合格率
by_dept = berkeley.pivot_table(index="学部", columns="性別", values=["出願","合格"], aggfunc="sum")
rate = (by_dept["合格"] / by_dept["出願"] * 100).round(1)
rate["女が高い?"] = np.where(rate["女"] > rate["男"], "← 女性のほうが高い", "")
rate

**6学部中4学部で、女性のほうが合格率が高い。** 残り2つも差はわずかである。

全体では女性が9ポイント低いのに、どの学部を見ても女性が不利ではない。なぜこうなるのか。

In [ ]:
# 出願先の分布を見る（女性はどの学部に出願していたか）
apply = berkeley.pivot_table(index="学部", columns="性別", values="出願", aggfunc="sum")
apply["学部全体の合格率"] = (by_dept["合格"].sum(axis=1) / by_dept["出願"].sum(axis=1) * 100).round(1)
apply["女性の出願割合"] = (apply["女"] / (apply["男"] + apply["女"]) * 100).round(1)
apply.sort_values("学部全体の合格率", ascending=False)

答えが出た。**合格率の高い学部A・Bには男性が集中し、合格率の低い学部C・E・Fに女性が集中している。**

学部Aの合格率は64%、学部Fは6%。**そもそも入りやすさが10倍違う。**
女性は難関の学部に多く出願していたので、全体をまぜると合格率が下がって見えた。

> **全体の数字は「差別」、分けると「逆」。どちらが本当かは、分けて初めてわかる。**

これを **シンプソンのパラドックス** という。

### なぜ起こるのか（ひとことで）

```
グループごとに「人数の偏り」や「別の要因」があると、
全体の平均は、その偏りに引っぱられて本当の傾向を隠す（時に逆転させる）
```

平均や全体は、**たった1個にまとめた数字**である。まとめる過程で、グループの情報が消える。

### 数字を見たときの合言葉

1. これ、**何かで分けたら違う話にならないか**（性別・学年・地域・時期・グループ）
2. どこかの**グループが極端に多く／少なく**混じっていないか

---
## 2. クロス集計表を自分で作る

使う関数は `pd.crosstab`。**行に置く項目**と**列に置く項目**を渡すだけ。

In [ ]:
# 学部 × 一人暮らし
pd.crosstab(df["学部"], df["一人暮らし"])

人数の表ができた。ただし **人数のままでは比べられない。**
学部ごとに在籍数が違うので、「経済学部のほうが一人暮らしが多い」と言っても、単に経済学部の人数が多いだけかもしれない。

**割合に直す。** `normalize="index"` で、行ごとに合計100%にする。

In [ ]:
# 行ごとの割合（%）にする
tab = pd.crosstab(df["学部"], df["一人暮らし"], normalize="index") * 100
tab.round(1)

In [ ]:
# 合計も一緒に出す（報告書にはこの形で載せる）
pd.crosstab(df["学部"], df["一人暮らし"], margins=True, margins_name="合計")

> **どちらの方向で割ったかを必ず書く。**
> `normalize="index"` は行方向（学部ごとに100%）、`normalize="columns"` は列方向（一人暮らしの人の中で100%）。
> 同じ表から、まったく違う主張が作れてしまう。

---
## 3. 層別を1つ加える ―― 結論は変わるか

バークレーでやったのと同じことを、自分のデータでやる。**3つめの項目で分けてみる。**

In [ ]:
# まず、分けずに見る：一人暮らしかどうかで、出席率の平均に差はあるか
print("【全体】一人暮らし別の出席率")
print(df.groupby("一人暮らし")["出席率"].agg(["count", "mean"]).round(1))

In [ ]:
# 次に、学部で分けて見る
print("【学部で分けた場合】")
split = df.pivot_table(index="学部", columns="一人暮らし", values="出席率", aggfunc="mean").round(1)
split["差（はい−いいえ）"] = (split["はい"] - split["いいえ"]).round(1)
split

**自分の目で確かめること。** 全体で見えた差は、学部ごとに分けても同じ向きに残っているか。消えているか。逆転しているか。

今回のデータでは劇的な逆転は起きない。**それも結果である。**「分けても結論は変わらなかった」は、報告書に書く価値のある一文だ。確かめずに「差がある」と書くことだけが、してはいけないことである。

---
## 4. 自分の調査データでやる

In [ ]:
# ここから先は「自分の調査データ」でやる。
# Google Forms の回答 → スプレッドシート → ファイル → ダウンロード → CSV で書き出したものを使う。
#
# 左のフォルダアイコン（📁）にCSVをドラッグしてから、ファイル名を書き換えて実行する。
# ※ 数字を手で打ち直さないこと。転記した瞬間に、それは元データではなくなる。

# mydf = pd.read_csv("自分のファイル名.csv")
# mydf.head()

In [ ]:
# クロス集計（列名を自分のものに書き換える）
# pd.crosstab(mydf["行にする項目"], mydf["列にする項目"], margins=True)

# 割合にする
# pd.crosstab(mydf["行にする項目"], mydf["列にする項目"], normalize="index").round(3) * 100

---
## 5. 卒業課題 ―― AIと突き合わせる

同じクロス集計表を **AIに直接作らせて**、自分のコードの結果と一致するか確かめる。

ずれる原因でいちばん多いのは、**空欄の数え方**である。pandasの `crosstab` は空欄の行を自動で除くが、AIが空欄を「その他」として数えていることがある。
**分母が違えば、割合は違う。**

---
## 課題7（8点）

**このノートブック（コードと出力）** ＋ **解釈3文**。

- [ ] 自分のデータのクロス集計表（人数と割合の両方）
- [ ] 層別を1つ加えた表
- [ ] 解釈3文。**うち1文は「層別したら何が変わったか」**（変わらなかったなら、そう書く）
- [ ] AIの結果と一致したか。ずれた場合は原因

提出期限：次回授業の開始まで（遅れた場合は50%）